# LLM-Based Text Framing Analysis

This notebook uses the Daria + Stephen human consensus labels as a calibration set for local LLM sentence labeling. Human consensus labels of `Unsure` are excluded from calibration because they do not provide a substantive framing judgment. The workflow has two correction mechanisms:

1. Human labels are inserted into the prompt as few-shot examples, so the local model follows the project codebook more closely.
2. Exact sentences in the usable human gold set are overridden with the human consensus label in the final corrected output.

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'AI-Policy' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import llm_framing
importlib.reload(llm_framing)

from llm_framing import (
    apply_human_gold_correction,
    classify_sentences_with_ollama,
    evaluate_against_gold,
    load_gold_labels,
    load_policy_sentences,
    make_few_shot_examples,
    summarize_country_framing,
)

def relpath(path):
    return Path(path).resolve().relative_to(REPO_ROOT.resolve()).as_posix()

pd.set_option('display.max_colwidth', 140)


## Configuration

In [ ]:
MODEL = 'deepseek-r1:14b'
OLLAMA_HOST = 'http://localhost:11434'

EXTRACTED_DIR = REPO_ROOT / 'data' / 'pdf' / 'AI Policy' / '_extracted'
GOLD_PATH = REPO_ROOT / 'data' / 'Human expert framing labels' / 'processed_data' / 'human_expert_framing_daria_stephen_agreement_agreed_sentences.csv'
OUTPUT_DIR = REPO_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Keep this small for a smoke test. Set to None to classify the full corpus.
RUN_LIMIT = None
USE_CACHE = True
CALIBRATION_TAG = 'daria_stephen_no_unsure'
RUN_TAG = 'full' if RUN_LIMIT is None else f'limit_{RUN_LIMIT}'
OUTPUT_TAG = f'{CALIBRATION_TAG}_{RUN_TAG}'

RAW_MODEL_PATH = OUTPUT_DIR / f'deepseek_ai_framing_sentence_labels_raw_{OUTPUT_TAG}.csv'
CORRECTED_PATH = OUTPUT_DIR / f'deepseek_ai_framing_sentence_labels_corrected_{OUTPUT_TAG}.csv'
SUMMARY_PATH = OUTPUT_DIR / f'deepseek_ai_framing_summary_corrected_{OUTPUT_TAG}.csv'
METRICS_PATH = OUTPUT_DIR / f'deepseek_ai_framing_gold_metrics_{OUTPUT_TAG}.csv'
CONFUSION_PATH = OUTPUT_DIR / f'deepseek_ai_framing_gold_confusion_{OUTPUT_TAG}.csv'

print(f'Model: {MODEL}')
print(f'Run tag: {RUN_TAG}')
print(f'Calibration tag: {CALIBRATION_TAG}')
print(f'Gold labels: {relpath(GOLD_PATH)}')
print(f'Extracted text: {relpath(EXTRACTED_DIR)}')
print(f'Output dir: {relpath(OUTPUT_DIR)}')


Model: deepseek-r1:14b
Run tag: limit_80
Calibration tag: daria_stephen_no_unsure
Gold labels: data/Human expert framing labels/processed_data/human_expert_framing_daria_stephen_agreement_agreed_sentences.csv
Extracted text: data/pdf/AI Policy/_extracted
Output dir: outputs


## Load Human Gold Labels and Policy Sentences

In [3]:
raw_gold = pd.read_csv(GOLD_PATH)
gold = load_gold_labels(GOLD_PATH, excluded_labels=['Unsure'])
sentences = load_policy_sentences(EXTRACTED_DIR)

print(f'Raw Daria + Stephen consensus sentences: {len(raw_gold)}')
print(f'Usable calibration sentences after excluding Unsure: {len(gold)}')
print(f'Excluded Unsure sentences: {(raw_gold["majority_label"] == "Unsure").sum()}')
print(f'Policy sentences available: {len(sentences)}')

gold[['item_id', 'country', 'sentence', 'gold_label', 'model_label_reference', 'majority_matches_reference']].head()


Raw Daria + Stephen consensus sentences: 23
Usable calibration sentences after excluding Unsure: 19
Excluded Unsure sentences: 4
Policy sentences available: 3730


,item_id,country,sentence,gold_label,model_label_reference,majority_matches_reference
0,S01,Canada,"Its generative Al tool, AgPal Chat, helps users find relevant funding and resources faster, supporting the sustainable growth and compet...",Innovation-oriented,AI-friendly,True
3,S06,Japan,"In the field of Al, where basic research and social implementation are closely connected, the lack of progress in implementation has bec...",Innovation-oriented,AI-friendly,True
4,S07,United Kingdom,oxinote 3) Setting a short-term target to train tens of thousands of Al professionals by 2030 will help bridge the estimated gap between...,Innovation-oriented,AI-friendly,True
5,S10,China,"Innovation is “fl owing water”, making China’s seed of advanced ideas into towering trees.",Innovation-oriented,AI-friendly,True
6,S11,Canada,Respondents suggested regular ethical audits to ensure compliance with the guidelines and prevent harmful biases.,Risk-oriented,AI-cautious,True


In [4]:
gold['gold_label'].value_counts().rename('gold_sentence_count').reset_index()


,gold_label,gold_sentence_count
0,Innovation-oriented,6
1,Mixed,6
2,Risk-oriented,5
3,Neutral,2


## Build Human-Calibrated Prompt Examples

In [5]:
examples = make_few_shot_examples(gold, max_examples_per_label=2)

print(f'Few-shot examples: {len(examples)}')
pd.DataFrame(examples)


Few-shot examples: 8


,sentence,label
0,"Its generative Al tool, AgPal Chat, helps users find relevant funding and resources faster, supporting the sustainable growth and compet...",Innovation-oriented
1,"In the field of Al, where basic research and social implementation are closely connected, the lack of progress in implementation has bec...",Innovation-oriented
2,Tailor-made treatment should be reserved for public procurement involving the processing of sensitive data.,Risk-oriented
3,They include principles for responsible stewardship of trustworthy AI and policy and cooperation recommendations.,Risk-oriented
4,"Some law or policy is silent on areas needed to govern AI, while others may introduce unintended bias in data collection or obstruct AI ...",Mixed
5,(V) Developing AI + Developing Governance 1.,Mixed
6,"These agencies could then provide analysis of AI adoption, job creation, displacement, and wage effects.",Neutral
7, The public reports of the Court of Auditors are available online on the website of the Court and the regional and territorial chambers...,Neutral


## Run or Load Local DeepSeek Labels

Make sure Ollama is available locally before running this cell. If you already pulled the model with `ollama run deepseek-r1:14b`, the API should use the same model name. If this is the first full run, set `RUN_LIMIT = None` in the configuration cell.

In [6]:
if USE_CACHE:
    raw_model_labels = classify_sentences_with_ollama(
        sentences,
        model=MODEL,
        host=OLLAMA_HOST,
        examples=examples,
        limit=RUN_LIMIT,
        checkpoint_path=RAW_MODEL_PATH,
        resume=True,
        timeout=600,
        max_retries=1,
        save_errors_as_unsure=True,
    )
    print(f'Saved or resumed raw model labels: {relpath(RAW_MODEL_PATH)}')
else:
    raw_model_labels = classify_sentences_with_ollama(
        sentences,
        model=MODEL,
        host=OLLAMA_HOST,
        examples=examples,
        limit=RUN_LIMIT,
        timeout=600,
        max_retries=1,
        save_errors_as_unsure=True,
    )
    raw_model_labels.to_csv(RAW_MODEL_PATH, index=False)
    print(f'Saved raw model labels: {relpath(RAW_MODEL_PATH)}')

print(f'Model label rows: {len(raw_model_labels)}')
raw_model_labels[['country', 'sentence_id', 'sentence', 'model_label', 'model_confidence', 'model_reason']].head()


loaded checkpoint with 80 completed sentences
classifying 0 remaining sentences (80/80 already done)
Saved or resumed raw model labels: outputs/deepseek_ai_framing_sentence_labels_raw_daria_stephen_no_unsure_limit_80.csv
Model label rows: 80


,country,sentence_id,sentence,model_label,model_confidence,model_reason
0,Canada,Canada_00001,"or ; , +l Treasury Board of Canada _ Secrétariat du Conseil du Trésor C d [| ad Secretariat du Canada anid a Al Strategy for the Federal...",Neutral,0.95,"The sentence is descriptive and administrative, listing publication details without any clear innovation-oriented or risk-oriented framing."
1,Canada,Canada_00002,Aussi offert en frangais sous le titre : Stratégie en matiére d’intelligence artificielle pour la fonction publique fédérale 2025-2027 A...,Risk-oriented,0.85,"Mentions responsible AI adoption, which relates to risk management."
2,Canada,Canada_00003,Overview Priority areas Expectations for federal organizations Relevant strategies and policy Engagement and consultation Overview On th...,Neutral,0.80,The sentence is descriptive and outlines sections or topics without indicating a clear innovation-oriented or risk-oriented framing.
3,Canada,Canada_00004,"Vision Principles scope Foreword from the President, Chief Information Officer and Chief Data Officer Why an Al Strategy?",Neutral,0.80,"The sentence appears to be an introduction or title from a document, setting the stage for discussing AI strategy without taking a clear..."
4,Canada,Canada_00005,Artificial intelligence (AI) presents unprecedented and wide-ranging opportunities to enhance the public service and the services it off...,Innovation-oriented,0.95,"The sentence highlights opportunities for enhancing public services through AI, indicating an innovation-oriented perspective."


## Apply Human Gold Correction

In [7]:
corrected_labels = apply_human_gold_correction(raw_model_labels, gold)
corrected_labels.to_csv(CORRECTED_PATH, index=False)

print(f'Saved corrected labels: {relpath(CORRECTED_PATH)}')
print(f'Rows corrected by exact human gold override: {corrected_labels["label_changed_by_human_gold"].sum()}')

corrected_labels[[
    'country', 'sentence_id', 'sentence', 'model_label', 'gold_label',
    'corrected_label', 'human_gold_override', 'label_changed_by_human_gold'
]].head()


Saved corrected labels: outputs/deepseek_ai_framing_sentence_labels_corrected_daria_stephen_no_unsure_limit_80.csv
Rows corrected by exact human gold override: 0


,country,sentence_id,sentence,model_label,gold_label,corrected_label,human_gold_override,label_changed_by_human_gold
0,Canada,Canada_00001,"or ; , +l Treasury Board of Canada _ Secrétariat du Conseil du Trésor C d [| ad Secretariat du Canada anid a Al Strategy for the Federal...",Neutral,NaN,Neutral,False,False
1,Canada,Canada_00002,Aussi offert en frangais sous le titre : Stratégie en matiére d’intelligence artificielle pour la fonction publique fédérale 2025-2027 A...,Risk-oriented,NaN,Risk-oriented,False,False
2,Canada,Canada_00003,Overview Priority areas Expectations for federal organizations Relevant strategies and policy Engagement and consultation Overview On th...,Neutral,NaN,Neutral,False,False
3,Canada,Canada_00004,"Vision Principles scope Foreword from the President, Chief Information Officer and Chief Data Officer Why an Al Strategy?",Neutral,NaN,Neutral,False,False
4,Canada,Canada_00005,Artificial intelligence (AI) presents unprecedented and wide-ranging opportunities to enhance the public service and the services it off...,Innovation-oriented,NaN,Innovation-oriented,False,False


## Gold-Set Evaluation

In [8]:
evaluation = evaluate_against_gold(corrected_labels)
metrics = evaluation['metrics']
confusion = evaluation['confusion']

metrics.to_csv(METRICS_PATH, index=False)
confusion.to_csv(CONFUSION_PATH, index=False)

print(f'Saved metrics: {relpath(METRICS_PATH)}')
print(f'Saved confusion table: {relpath(CONFUSION_PATH)}')
metrics


Saved metrics: outputs/deepseek_ai_framing_gold_metrics_daria_stephen_no_unsure_limit_80.csv
Saved confusion table: outputs/deepseek_ai_framing_gold_confusion_daria_stephen_no_unsure_limit_80.csv


,gold_sentences,raw_model_accuracy,corrected_accuracy,human_overrides
0,1,1.0,1.0,0


In [9]:
confusion


raw_model_label,gold_label,Innovation-oriented
0,Innovation-oriented,1


## Country-Level Corrected Framing Summary

In [10]:
country_summary = summarize_country_framing(corrected_labels, label_col='corrected_label')
country_summary.to_csv(SUMMARY_PATH, index=False)

print(f'Saved corrected country summary: {relpath(SUMMARY_PATH)}')
country_summary


Saved corrected country summary: outputs/deepseek_ai_framing_summary_corrected_daria_stephen_no_unsure_limit_80.csv


,country,Innovation-oriented,Mixed,Neutral,Risk-oriented,Unsure,total_sentences,net_framing_score,mean_framing_score,innovation_to_risk_ratio
0,Canada,24,10,26,20,0,80,4.0,0.05,1.195122


## Visualize Corrected Country Framing Summary

In [ ]:
import matplotlib.pyplot as plt

plot_df = country_summary.copy()
if plot_df.empty and SUMMARY_PATH.exists():
    plot_df = pd.read_csv(SUMMARY_PATH)

label_cols = ['Innovation-oriented', 'Risk-oriented', 'Mixed', 'Neutral', 'Unsure']
plot_df = plot_df.sort_values('mean_framing_score', ascending=True).reset_index(drop=True)
share_df = (
    plot_df.set_index('country')[label_cols]
    .div(plot_df.set_index('country')['total_sentences'], axis=0)
    .mul(100)
)

colors = {
    'Innovation-oriented': '#2ca02c',
    'Risk-oriented': '#d62728',
    'Mixed': '#9467bd',
    'Neutral': '#7f7f7f',
    'Unsure': '#bcbd22',
}

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(15, 6),
    gridspec_kw={'width_ratios': [1.7, 1]},
)

left = pd.Series(0.0, index=share_df.index)
for col in label_cols:
    values = share_df[col]
    ax1.barh(share_df.index, values, left=left, color=colors[col], label=col)
    left += values

ax1.set_xlabel('Share of corrected sentence labels (%)')
ax1.set_title('Corrected framing label composition by country')
ax1.set_xlim(0, 100)
ax1.grid(axis='x', alpha=0.25)
ax1.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)

bar_colors = ['#2ca02c' if v >= 0 else '#d62728' for v in plot_df['mean_framing_score']]
ax2.barh(plot_df['country'], plot_df['mean_framing_score'], color=bar_colors)
ax2.axvline(0, color='#444', linewidth=1)
ax2.set_xlabel('Mean framing score')
ax2.set_title('Net innovation vs risk orientation')
ax2.grid(axis='x', alpha=0.25)
for _, row in plot_df.iterrows():
    x = row['mean_framing_score']
    offset = 0.015 if x >= 0 else -0.015
    ha = 'left' if x >= 0 else 'right'
    ax2.text(x + offset, row['country'], f"{x:.2f}", va='center', ha=ha, fontsize=9)

fig.suptitle(f'DeepSeek corrected framing summary ({OUTPUT_TAG})', fontsize=14, fontweight='bold')
fig.tight_layout()
figure_path = OUTPUT_DIR / f'deepseek_ai_framing_country_summary_corrected_{OUTPUT_TAG}.png'
fig.savefig(figure_path, dpi=200, bbox_inches='tight')
print(f'Saved corrected country summary figure: {relpath(figure_path)}')
plt.show()


## How Human Labels Correct AI Labeling

- The prompt uses Daria + Stephen substantive consensus labels as examples, excluding `Unsure` cases.
- The final output keeps both `model_label` and `corrected_label`. For any sentence that appears in the usable human gold set, `corrected_label` is forced to the human consensus.
- Human `Unsure` labels are retained in the source consensus file as diagnostics, but they are not used for few-shot calibration, exact-match override, or gold-set evaluation.
- The metrics and confusion table show where the local model disagrees with the usable human gold labels. Use those disagreements to add more annotation examples or refine the codebook.